# BestBuy Full Pipeline Test (curl_cffi + APIs)

Test full pipeline giong skipflow.ipynb nhung dung curl_cffi + BestBuy APIs thay vi Playwright.

Pipeline:
1. Search BestBuy (curl_cffi search page + Apollo cache) -> skuIds
2. Filter sale items (priceBlocks API - batch)
3. Scrape product details (v2 API - features, URL)
4. Select top 5 deals (GPT-5-mini)
5. Estimate prices (EnsembleAgent - 3 models)
6. Final results table

In [1]:
# Cell 1: Setup & Imports
import os
import sys
import logging

os.chdir('/home/hieu0606sunny/price2026wsl/tech2ai/segment4')
sys.path.insert(0, '/home/hieu0606sunny/price2026wsl/tech2ai/segment4')

logging.basicConfig(level=logging.INFO)
logging.getLogger().setLevel(logging.INFO)

from dotenv import load_dotenv
load_dotenv(override=True)

print(f'Working directory: {os.getcwd()}')
print('Setup complete!')

Working directory: /home/hieu0606sunny/price2026wsl/tech2ai/segment4
Setup complete!


In [2]:
# Cell 2: Import pipeline components
import chromadb
from price_agents.bestbuy_deals import ScrapedBestBuyDeal
from price_agents.bestbuy_scanner_agent import BestBuyScannerAgent
from price_agents.deals import Deal, DealSelection, Opportunity
from price_agents.ensemble_agent import EnsembleAgent

# Import buoc1 functions
sys.path.insert(0, 'base/fix_bestbuy_tocdo_thang3')
from buoc1 import search_bestbuy, get_price_blocks, get_product_details
from curl_cffi import requests as curl_requests

print('Pipeline components imported!')
print('  - search_bestbuy (curl_cffi search page)')
print('  - get_price_blocks (batch API)')
print('  - get_product_details (v2 API - features + URL)')
print('  - BestBuyScannerAgent (GPT-5-mini)')
print('  - EnsembleAgent (3 models)')

INFO:datasets:PyTorch version 2.9.0 available.


Pipeline components imported!
  - search_bestbuy (curl_cffi search page)
  - get_price_blocks (batch API)
  - get_product_details (v2 API - features + URL)
  - BestBuyScannerAgent (GPT-5-mini)
  - EnsembleAgent (3 models)


In [3]:
# Cell 3: Initialize EnsembleAgent (takes time, run once)
print('Initializing EnsembleAgent (3 models)...')
print('=' * 60)

DB_PATH = 'products_vectorstore'
client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_or_create_collection('products')
print(f'ChromaDB: {collection.count()} documents')

ensemble = EnsembleAgent(collection)
print('\nEnsembleAgent ready!')

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


Initializing EnsembleAgent (3 models)...


INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Specialist Agent] Specialist Agent is ready
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI


ChromaDB: 800000 documents


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using cuda
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded
INFO:root:[Ensemble Agent] Ensemble Agent is ready



EnsembleAgent ready!


In [4]:
# Cell 4: User Input
TEST_KEYWORD = 'laptop'

print('=' * 60)
print('BestBuy Pipeline - curl_cffi + APIs')
print('=' * 60)
print(f'\nKeyword: \'{TEST_KEYWORD}\'')

BestBuy Pipeline - curl_cffi + APIs

Keyword: 'laptop'


In [5]:
# Cell 5: Step 1 - Search BestBuy (curl_cffi search page)
import time

print('=' * 60)
print(f'STEP 1: Search BestBuy for \'{TEST_KEYWORD}\'')
print('=' * 60)

start = time.time()

# Init curl_cffi session + bypass country selection
session = curl_requests.Session(impersonate='chrome')
session.get('https://www.bestbuy.com/?intl=nosplash', timeout=15)

# Search -> lay skuIds tu Apollo cache
apollo_products = search_bestbuy(session, TEST_KEYWORD)

elapsed = time.time() - start
print(f'\nFound {len(apollo_products)} SKUs in {elapsed:.1f}s')

STEP 1: Search BestBuy for 'laptop'


INFO:buoc1:[Step 1] Search: https://www.bestbuy.com/site/searchpage.jsp?st=laptop
INFO:buoc1:  Status: 200 | Size: 1,877,708 bytes
INFO:buoc1:  Found 118 unique SKUs (4 with URLs)



Found 118 SKUs in 3.5s


In [6]:
# Cell 6: Step 2 - Filter sale items (priceBlocks API)
print('=' * 60)
print(f'STEP 2: Filter sale items ({len(apollo_products)} SKUs)')
print('=' * 60)

start = time.time()

sku_ids = [p['skuId'] for p in apollo_products]
price_data = get_price_blocks(session, sku_ids)

# Filter ON SALE only
sale_skus = [sku for sku, data in price_data.items() if data['onSale']]

elapsed = time.time() - start
print(f'\nResults ({elapsed:.1f}s):')
print(f'  Total with price data: {len(price_data)}')
print(f'  ON SALE: {len(sale_skus)}')
print()
for i, sku in enumerate(sale_skus, 1):
    pd = price_data[sku]
    print(f'  {i}. [{sku}] ${pd["currentPrice"]} (was ${pd["regularPrice"]}, save ${pd["savingsAmount"]}) - {pd["name"][:60]}...')

if not sale_skus:
    print('\nNo sale items found. Using all products instead.')
    sale_skus = list(price_data.keys())[:10]

INFO:buoc1:[Step 2] priceBlocks: 118 SKUs


STEP 2: Filter sale items (118 SKUs)


INFO:buoc1:  Status: 200
INFO:buoc1:  Got price data for 9/118 SKUs



Results (1.8s):
  Total with price data: 9
  ON SALE: 5

  1. [6612253] $579.99 (was $799.99, save $220.0) - HP - 15.6" Full HD Touch-Screen Laptop - Intel Core i7 - 16G...
  2. [6612976] $179.99 (was $219.99, save $40.0) - HP - 14" Laptop - Intel Processor N150 - 4GB Memory - 128GB ...
  3. [6636629] $299.99 (was $529.99, save $230.0) - Lenovo - IdeaPad 1 15.6" Full HD Laptop - AMD Ryzen 5 7520U ...
  4. [6623881] $799.99 (was $1009.99, save $210.0) - HP - Victus 15.6" 144Hz Full HD Gaming Laptop - AMD Ryzen 7 ...
  5. [6609085] $139.0 (was $279.0, save $140.0) - ASUS - CX14 14" FHD Chromebook Laptop - Intel Celeron - 4GB ...


In [7]:
# Cell 7: Step 3 - Scrape product details (v2 API - features + URL)
# Lay top 10 sale products
sale_skus = sale_skus[:10]

print('=' * 60)
print(f'STEP 3: Fetch features for {len(sale_skus)} sale products')
print('=' * 60)

start = time.time()

# Build ScrapedBestBuyDeal objects
scraped_deals = []
for i, sku in enumerate(sale_skus, 1):
    pd = price_data[sku]
    details = get_product_details(session, sku)
    
    deal = ScrapedBestBuyDeal(
        title=pd['name'],
        brand=pd['brand'],
        price=pd['currentPrice'],
        features=details['features'] or pd['name'],
        url=details['url'] or f'https://www.bestbuy.com/site/{sku}.p',
    )
    scraped_deals.append(deal)
    print(f'  [{i}/{len(sale_skus)}] {deal.title[:60]}... | ${deal.price}')

elapsed = time.time() - start
print(f'\nScraped {len(scraped_deals)} products in {elapsed:.1f}s')

STEP 3: Fetch features for 5 sale products
  [1/5] HP - 15.6" Full HD Touch-Screen Laptop - Intel Core i7 - 16G... | $579.99
  [2/5] HP - 14" Laptop - Intel Processor N150 - 4GB Memory - 128GB ... | $179.99
  [3/5] Lenovo - IdeaPad 1 15.6" Full HD Laptop - AMD Ryzen 5 7520U ... | $299.99
  [4/5] HP - Victus 15.6" 144Hz Full HD Gaming Laptop - AMD Ryzen 7 ... | $799.99
  [5/5] ASUS - CX14 14" FHD Chromebook Laptop - Intel Celeron - 4GB ... | $139.0

Scraped 5 products in 3.3s


In [8]:
# Cell 8: Step 4 - Select top 5 deals (GPT-5-mini)
print('=' * 60)
print(f'STEP 4: Select top 5 deals with GPT-5-mini')
print('=' * 60)

start = time.time()

scanner = BestBuyScannerAgent()
deal_selection = scanner.scan(scraped_deals)

elapsed = time.time() - start

if deal_selection and deal_selection.deals:
    print(f'\nSelected {len(deal_selection.deals)} best deals in {elapsed:.1f}s:')
    for i, deal in enumerate(deal_selection.deals, 1):
        print(f'\n  [{i}] {deal.product_description[:80]}...')
        print(f'      ${deal.price}')
else:
    print('\nNo deals selected. Check scraped_deals.')

INFO:root:[BestBuy Scanner Agent] BestBuy Scanner Agent is initializing
INFO:root:[BestBuy Scanner Agent] BestBuy Scanner Agent is ready
INFO:root:[BestBuy Scanner Agent] Calling gpt-5-mini with 5 deals...


STEP 4: Select top 5 deals with GPT-5-mini


INFO:root:[BestBuy Scanner Agent] Selected 5 deals



Selected 5 best deals in 22.9s:

  [1] This HP 15.6-inch laptop is powered by a 13th Generation Intel Core i7-1355U pro...
      $579.99

  [2] This HP 14-inch laptop uses an Intel N150 processor with integrated Intel graphi...
      $179.99

  [3] The Lenovo IdeaPad 1 is a 15.6-inch Full HD laptop powered by an AMD Ryzen 5 752...
      $299.99

  [4] The HP Victus is a 15.6-inch gaming laptop built around an AMD Ryzen 7 7445HS pr...
      $799.99

  [5] The ASUS CX14 is a 14-inch Chromebook powered by an Intel Celeron N4500 processo...
      $139.0


In [9]:
# Cell 9: Step 5 - Estimate prices (EnsembleAgent - 3 models)
print('=' * 60)
print(f'STEP 5: Estimate prices with EnsembleAgent')
print('=' * 60)

start = time.time()
opportunities = []

for i, deal in enumerate(deal_selection.deals, 1):
    print(f'\n[{i}/{len(deal_selection.deals)}] Estimating: {deal.product_description[:50]}...')
    
    estimate = ensemble.price(deal.product_description)
    discount = estimate - deal.price
    
    opportunity = Opportunity(
        deal=deal,
        estimate=estimate,
        discount=discount
    )
    opportunities.append(opportunity)
    
    print(f'  Sale: ${deal.price:.2f} | Estimate: ${estimate:.2f} | Discount: ${discount:.2f}')

elapsed = time.time() - start
print(f'\nEstimation done in {elapsed:.1f}s')

INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
22:00:10 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


STEP 5: Estimate prices with EnsembleAgent

[1/5] Estimating: This HP 15.6-inch laptop is powered by a 13th Gene...


22:00:10 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $799.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $729.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $774.87
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $740.59
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
22:00:48 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


...

  Sale: $799.99 | Estimate: $1018.89 | Discount: $218.90

[5/5] Estimating: The ASUS CX14 is a 14-inch Chromebook powered by a...


22:00:58 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $189.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $210.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $212.61
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $208.16


  Sale: $139.00 | Estimate: $208.16 | Discount: $69.16

Estimation done in 51.5s


In [10]:
# Cell 10: Final Results Table
print('=' * 80)
print('FINAL RESULTS - Sorted by Discount')
print('=' * 80)
print(f'\nKeyword: \'{TEST_KEYWORD}\'')
print(f'Deals: {len(opportunities)}\n')

# Sort by discount descending
opportunities.sort(key=lambda x: x.discount, reverse=True)

for i, opp in enumerate(opportunities, 1):
    discount_pct = (opp.discount / opp.estimate * 100) if opp.estimate > 0 else 0
    
    if opp.discount > 200:
        status = 'HOT DEAL'
    elif opp.discount > 100:
        status = 'Good Deal'
    elif opp.discount > 0:
        status = 'OK'
    else:
        status = 'Overpriced'
    
    print(f'--- #{i} [{status}] ---')
    print(f'  Product:  {opp.deal.product_description[:80]}...')
    print(f'  Sale:     ${opp.deal.price:.2f}')
    print(f'  Estimate: ${opp.estimate:.2f}')
    print(f'  Discount: ${opp.discount:.2f} ({discount_pct:.0f}%)')
    print(f'  URL:      {opp.deal.url[:80]}')
    print()

FINAL RESULTS - Sorted by Discount

Keyword: 'laptop'
Deals: 5

--- #1 [HOT DEAL] ---
  Product:  The HP Victus is a 15.6-inch gaming laptop built around an AMD Ryzen 7 7445HS pr...
  Sale:     $799.99
  Estimate: $1018.89
  Discount: $218.90 (21%)
  URL:      https://www.bestbuy.com/product/hp-victus-15-6-144hz-full-hd-gaming-laptop-amd-r

--- #2 [Good Deal] ---
  Product:  The Lenovo IdeaPad 1 is a 15.6-inch Full HD laptop powered by an AMD Ryzen 5 752...
  Sale:     $299.99
  Estimate: $483.94
  Discount: $183.95 (38%)
  URL:      https://www.bestbuy.com/product/lenovo-ideapad-1-15-6-full-hd-laptop-amd-ryzen-5

--- #3 [Good Deal] ---
  Product:  This HP 15.6-inch laptop is powered by a 13th Generation Intel Core i7-1355U pro...
  Sale:     $579.99
  Estimate: $740.59
  Discount: $160.60 (22%)
  URL:      https://www.bestbuy.com/product/hp-15-6-full-hd-touch-screen-laptop-intel-core-i

--- #4 [Good Deal] ---
  Product:  This HP 14-inch laptop uses an Intel N150 processor with integra